# Прогноз цены авто — эталон (исправленная версия)

Что исправлено против сдачи ученика:
1. **Baseline** `DummyRegressor` — есть с чем сравнивать.
2. **Pipeline** (`ColumnTransformer` + `RandomForest`) — стандарт курса.
3. **Утечка убрана** — импутация внутри Pipeline, обучается только на train.
4. **Добавлены категориальные признаки** (марка/топливо/коробка) — главный резерв R².
5. **Gradio-интерфейс** + заготовка README.

In [ ]:
import pandas as pd

df = pd.read_csv("cars.csv")

# оставляем строки, где есть цена (это цель, её заполнять нельзя)
df = df.dropna(subset=["price"])

print("Строк:", len(df))
print("Колонки:", df.columns.tolist())

In [ ]:
# ✅ ФИКС 4: разделяем признаки на числовые и категориальные.
# Отредактируй списки под реальные колонки своего CSV (проверь по df.columns выше).
numeric_features = ["year", "mileage", "engine_volume"]
categorical_features = ["brand", "fuel", "transmission"]   # <- добавь то, что есть в данных

# берём только те колонки, что реально существуют (чтобы не падало)
numeric_features = [col for col in numeric_features if col in df.columns]
categorical_features = [col for col in categorical_features if col in df.columns]

features = numeric_features + categorical_features
X = df[features]
y = df["price"]

print("Числовые:", numeric_features)
print("Категориальные:", categorical_features)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Обучение:", len(X_train))
print("Тест:", len(X_test))

In [ ]:
# ✅ ФИКС 1: BASELINE — "отметка на стене": всегда предсказывает среднюю цену.
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, r2_score

baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)
base_pred = baseline.predict(X_test)

print("BASELINE")
print("  MAE:", round(mean_absolute_error(y_test, base_pred)), "сомони")
print("  R²:", round(r2_score(y_test, base_pred), 3))

In [ ]:
# ✅ ФИКС 2 + ФИКС 3: Pipeline с ColumnTransformer.
# Импутация и OneHot считаются ТОЛЬКО на train -> утечки нет.
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor

numeric_pipe = SimpleImputer(strategy="median")

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features)
])

model = Pipeline([
    ("prep", preprocessor),
    ("rf", RandomForestRegressor(n_estimators=200, random_state=42))
])

model.fit(X_train, y_train)
print("Модель обучена!")

In [ ]:
pred = model.predict(X_test)

print("МОДЕЛЬ")
print("  MAE:", round(mean_absolute_error(y_test, pred)), "сомони")
print("  R²:", round(r2_score(y_test, pred), 3))
print()
print("Модель должна быть заметно лучше baseline по MAE — иначе она бесполезна.")

In [ ]:
# пример прогноза (реалистичные значения)
sample = {"year": 2015, "mileage": 120000, "engine_volume": 2.5}
# добавим категориальные, если они используются
for col in categorical_features:
    sample[col] = X_train[col].mode()[0]

car = pd.DataFrame([sample])[features]
price = model.predict(car)[0]
print("Предсказанная цена:", round(price), "сомони")

In [ ]:
# ✅ ФИКС 5: Gradio-интерфейс (веб-окно для родителей на Demo Day)
!pip install gradio -q

In [ ]:
import gradio as gr

def predict_price(year, mileage, engine_volume, *cat_values):
    row = {"year": year, "mileage": mileage, "engine_volume": engine_volume}
    for col, val in zip(categorical_features, cat_values):
        row[col] = val
    car = pd.DataFrame([row])[features]
    price = model.predict(car)[0]
    return f"Оценка: {round(price)} сомони"

inputs = [
    gr.Number(label="Год выпуска", value=2015),
    gr.Number(label="Пробег (км)", value=120000),
    gr.Number(label="Объём двигателя", value=2.5),
]
# выпадающие списки для категориальных признаков
for col in categorical_features:
    inputs.append(gr.Dropdown(sorted(X_train[col].dropna().unique().tolist()), label=col))

demo = gr.Interface(
    fn=predict_price,
    inputs=inputs,
    outputs=gr.Text(label="Прогноз цены"),
    title="Прогноз цены автомобиля",
    description="Введите параметры авто — модель оценит цену в сомони."
)

demo.launch(share=True)

## README (заготовка)

**Задача:** регрессия — предсказать цену б/у авто (сомони).

**Данные:** `cars.csv`, признаки: год, пробег, объём двигателя + категориальные.

**Baseline:** `DummyRegressor(mean)` — предсказывает среднюю цену.

**Модель:** `Pipeline(ColumnTransformer + RandomForestRegressor)`.

**Метрики:** MAE (главная), R². Модель сравнивается с baseline.

**Интерфейс:** Gradio (`launch(share=True)`).

**Запуск:** выполнить ячейки по порядку, загрузив `cars.csv`.